# NB12 - Capstone: Training Diagnostics Pipeline
## Background
Debugging a failed training run requires systematically ruling out each failure mode. Modern training dashboards (wandb, tensorboard) track dozens of signals, but knowing which signals matter and what they indicate is a skill separate from running the metrics. This notebook assembles all Series 30 concepts into a unified diagnostic system.

## What You'll Learn
- A full training run with AdamW + cosine schedule + optional SAM
- Real-time diagnostic signals: gradient norm, loss ratio, weight norm
- Grokking detection via delayed generalization monitoring
- Loss landscape sharpness tracking during training
- Automated health report: optimizer, schedule, gradient flow, sharpness

## Why This Matters
Professional ML engineers spend as much time diagnosing training failures as they do implementing models. A systematic diagnostic pipeline prevents wasted GPU hours and helps distinguish between: bad hyperparameters, wrong optimizer, architecture issues, and data problems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Callable

# ── AdamW with gradient clipping ──────────────────────────────────────────
@dataclass
class AdamW:
    lr: float = 1e-3
    beta1: float = 0.9
    beta2: float = 0.999
    eps: float = 1e-8
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    _m: Optional[np.ndarray] = field(default=None, init=False)
    _v: Optional[np.ndarray] = field(default=None, init=False)
    _t: int = field(default=0, init=False)

    def step(self, params: np.ndarray, grads: np.ndarray):
        # Gradient clipping
        grad_norm = np.linalg.norm(grads)
        if grad_norm > self.max_grad_norm:
            grads = grads * (self.max_grad_norm / grad_norm)
        if self._m is None:
            self._m = np.zeros_like(params)
            self._v = np.zeros_like(params)
        self._t += 1
        self._m = self.beta1*self._m + (1-self.beta1)*grads
        self._v = self.beta2*self._v + (1-self.beta2)*grads**2
        m_hat = self._m/(1-self.beta1**self._t)
        v_hat = self._v/(1-self.beta2**self._t)
        update = self.lr * m_hat/(np.sqrt(v_hat)+self.eps)
        return params*(1-self.lr*self.weight_decay) - update, grad_norm

    def reset(self): self._m = self._v = None; self._t = 0

# ── Cosine warmup schedule ────────────────────────────────────────────────
def warmup_cosine(step, total_steps, warmup_steps, lr_max, lr_min=1e-6):
    if step < warmup_steps:
        return lr_max * step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return lr_min + 0.5*(lr_max - lr_min)*(1 + np.cos(np.pi * progress))

# ── Training problem setup ────────────────────────────────────────────────
np.random.seed(42)
n_train, n_test = 25, 300
x_train = np.linspace(-1, 1, n_train)
x_test  = np.linspace(-1, 1, n_test)
y_train = np.sin(2*np.pi*x_train) + np.cos(np.pi*x_train) + 0.2*np.random.randn(n_train)
y_test  = np.sin(2*np.pi*x_test) + np.cos(np.pi*x_test)

degree = 12
X_train = np.stack([x_train**i for i in range(degree+1)], axis=-1)
X_test  = np.stack([x_test**i for i in range(degree+1)], axis=-1)

def loss_and_grad(w, X, y, fn=False):
    pred = X @ w; err = pred - y
    loss = np.mean(err**2)
    grad = 2 * X.T @ err / len(y)
    return loss, grad

print('Training problem: degree-12 polynomial regression, n_train=25')
print('Diagnostics to track: grad_norm, weight_norm, train_loss, test_loss, sharpness')

In [ ]:
# ── Full diagnostic training run ──────────────────────────────────────────
@dataclass
class TrainingDiagnostics:
    steps: List[int] = field(default_factory=list)
    train_losses: List[float] = field(default_factory=list)
    test_losses: List[float] = field(default_factory=list)
    grad_norms: List[float] = field(default_factory=list)
    weight_norms: List[float] = field(default_factory=list)
    learning_rates: List[float] = field(default_factory=list)
    loss_ratios: List[float] = field(default_factory=list)  # train/test

    def record(self, step, train_loss, test_loss, grad_norm, weight_norm, lr):
        self.steps.append(step)
        self.train_losses.append(train_loss)
        self.test_losses.append(test_loss)
        self.grad_norms.append(grad_norm)
        self.weight_norms.append(weight_norm)
        self.learning_rates.append(lr)
        self.loss_ratios.append(train_loss / (test_loss + 1e-10))

    def detect_grokking(self, patience: int = 200) -> Optional[int]:
        """Detect delayed generalization: train loss down but test loss still high."""
        if len(self.train_losses) < patience:
            return None
        for i in range(patience, len(self.train_losses)):
            tr_prev = np.mean(self.train_losses[i-patience:i-patience//2])
            te_prev = np.mean(self.test_losses[i-patience:i-patience//2])
            tr_now = np.mean(self.train_losses[i-patience//2:i])
            te_now = np.mean(self.test_losses[i-patience//2:i])
            # Grokking: train improved, test suddenly dropped
            if tr_now < tr_prev * 0.9 and te_now < te_prev * 0.9:
                return i
        return None

def run_diagnostic_training(n_steps=5000, lr_max=1e-2, weight_decay=0.05,
                              warmup=200, log_every=50):
    diag = TrainingDiagnostics()
    opt = AdamW(lr=lr_max, weight_decay=weight_decay, max_grad_norm=1.0)
    opt.reset()
    w = np.zeros(degree+1)

    for step in range(n_steps):
        lr = warmup_cosine(step, n_steps, warmup, lr_max)
        opt.lr = lr
        train_loss, grad = loss_and_grad(w, X_train, y_train)
        w, grad_norm = opt.step(w, grad)
        test_loss = loss_and_grad(w, X_test, y_test)[0]

        if step % log_every == 0:
            diag.record(step, train_loss, test_loss,
                        grad_norm, np.linalg.norm(w), lr)

    return w, diag

print('Running diagnostic training...')
w_final, diag = run_diagnostic_training(n_steps=5000)
grok_step = diag.detect_grokking(patience=5)
print(f'Training complete. Final train loss: {diag.train_losses[-1]:.4f}')
print(f'Final test loss: {diag.test_losses[-1]:.4f}')
if grok_step:
    print(f'Grokking-like generalization detected at step ~{grok_step * 50}')
else:
    print('No grokking pattern detected (gradual learning)')

In [ ]:
# ── Comprehensive diagnostic dashboard ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
steps = diag.steps

axes[0,0].semilogy(steps, diag.train_losses, label='Train')
axes[0,0].semilogy(steps, diag.test_losses, label='Test')
axes[0,0].set_title('Loss Curves'); axes[0,0].set_xlabel('Step')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].semilogy(steps, diag.grad_norms)
axes[0,1].axhline(1.0, color='red', linestyle='--', label='Clip threshold')
axes[0,1].set_title('Gradient Norm'); axes[0,1].set_xlabel('Step')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(steps, diag.weight_norms)
axes[0,2].set_title('Weight Norm ||w||'); axes[0,2].set_xlabel('Step')
axes[0,2].grid(True, alpha=0.3)

axes[1,0].plot(steps, diag.learning_rates)
axes[1,0].set_title('Learning Rate Schedule'); axes[1,0].set_xlabel('Step')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(steps, diag.loss_ratios)
axes[1,1].axhline(1.0, color='green', linestyle='--', label='Perfect generalization')
axes[1,1].set_title('Train/Test Loss Ratio'); axes[1,1].set_xlabel('Step')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(x_test, y_test, 'k--', linewidth=2, label='True')
axes[1,2].plot(x_test, X_test @ w_final, 'r-', linewidth=1.5, label='Predicted')
axes[1,2].scatter(x_train, y_train, c='k', s=20, zorder=5, label='Train data')
axes[1,2].set_title('Final Fit'); axes[1,2].legend(fontsize=8)
axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/s30_12_diagnostic_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Automated health report ───────────────────────────────────────────────
print('=' * 55)
print('  TRAINING DIAGNOSTICS REPORT')
print('=' * 55)
print()

# Gradient norm health
clipped_pct = sum(1 for g in diag.grad_norms if g > 0.9) / len(diag.grad_norms) * 100
print(f'[Gradient Norm]')
print(f'  Mean: {np.mean(diag.grad_norms):.4f}')
print(f'  Max:  {np.max(diag.grad_norms):.4f}')
print(f'  Clipped steps: {clipped_pct:.1f}%')
if clipped_pct > 50:
    print('  WARNING: Frequent gradient clipping - consider lower lr or larger warmup')
else:
    print('  OK: Gradient norms under control')
print()

# Loss ratio health
final_ratio = diag.loss_ratios[-1]
print(f'[Generalization]')
print(f'  Final train/test ratio: {final_ratio:.3f}')
if final_ratio < 0.5:
    print('  OK: Generalizing well (test loss < 2x train loss)')
elif final_ratio < 0.2:
    print('  WARNING: Moderate overfitting detected')
else:
    print('  ALERT: Significant overfitting - increase weight_decay or reduce model size')
print()

# Weight norm trend
wn_slope = (diag.weight_norms[-1] - diag.weight_norms[0]) / len(diag.weight_norms)
print(f'[Weight Norm]')
print(f'  Start: {diag.weight_norms[0]:.4f}, End: {diag.weight_norms[-1]:.4f}')
print(f'  Trend: {"increasing" if wn_slope > 0 else "decreasing"} ({wn_slope:+.5f}/step)')
print()

print(f'[Summary]')
print(f'  Optimizer: AdamW (wd=0.05, max_grad_norm=1.0)')
print(f'  Schedule: Warmup+Cosine (200 warmup, 5000 total)')
print(f'  Final test MSE: {diag.test_losses[-1]:.4f}')
print()
print('=== Series 30: Optimization & Training Dynamics Complete ===')
print()
print('Topics covered:')
topics = [
    'NB01: SGD & Momentum', 'NB02: Adam & AdamW', 'NB03: Modern Optimizers (Lion/Sophia/ADOPT)',
    'NB04: Learning Rate Schedules', 'NB05: Loss Landscape', 'NB06: SAM',
    'NB07: Gradient Flow & Vanishing Gradients', 'NB08: Grokking',
    'NB09: Double Descent', 'NB10: Neural Collapse', 'NB11: Mixed Precision',
    'NB12: Capstone Training Diagnostics',
]
for t in topics:
    print(f'  {t}')